In [ ]:
#### SimpleNN Training

This notebook trains SimpleNN models (simple feedforward neural networks) for quantum error correction.

**Purpose:** Baseline comparison against GNN-based decoders to determine if graph structure helps.

**Key Differences from GNN training:**
- Input: Flat syndrome arrays `[batch, num_detectors]`
- No graph structure - just raw syndrome measurements
- Uses `FlatDatasetCache` instead of `DatasetCache`

For hyperparameter tuning, see `code/nn/tuning.ipynb`.

In [ ]:
import sys
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('..')

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
from tqdm import tqdm
import gc

# Import from benchmark_models.py
from benchmark_models import (
    SurfaceCodeSampler,
    SimpleNNModel,
    SimpleNN,
    FlatDatasetCache,
    SimpleNNTrainer,
    count_logical_errors,
)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

#### Generate Flat Dataset Cache

Generate flat syndrome array datasets for SimpleNN training.
This only needs to be run once per distance.

In [ ]:
# ============================================================
# CONFIGURATION: Define which flat datasets to generate
# ============================================================

# Standard error rates (same as graph datasets for fair comparison)
STANDARD_P_VALUES = [0.001, 0.003, 0.005, 0.007]
STANDARD_P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Define datasets to generate
# Each entry: (name, distance, n_samples)
FLAT_DATASETS_TO_GENERATE = [
    ("d3_baseline", 3, 1_000_000),
    ("d5_baseline", 5, 1_000_000),
    ("d7_baseline", 7, 1_000_000),
    ("d9_baseline", 9, 1_000_000),
    # ("d11_baseline", 11, 1_000_000),
    # ("d13_baseline", 13, 1_000_000),
]

print("Flat datasets configured for generation:")
print(f"  Error rates: {STANDARD_P_VALUES}")
print(f"  Weights: {STANDARD_P_WEIGHTS}")
print()
for name, d, n in FLAT_DATASETS_TO_GENERATE:
    print(f"  • {name}: d={d}, n={n:,}")

In [ ]:
# ============================================================
# GENERATE FLAT DATASETS
# ============================================================
# This cell will generate all configured flat datasets.
# Skip any that already exist.

flat_datasets_dir = BASE_PATH / "datasets" / "flat"

for name, d, n_samples in FLAT_DATASETS_TO_GENERATE:
    # Check if dataset already exists
    if (flat_datasets_dir / f"{name}.pt").exists():
        print(f"⏭️  Flat dataset '{name}' already exists, skipping...")
        continue

    print(f"\n{'='*60}")
    print(f"Generating flat dataset: {name}")
    print(f"{'='*60}")

    # Create and generate dataset
    cache = FlatDatasetCache(base_path=BASE_PATH, device=device)
    cache.generate(
        d=d,
        n_samples=n_samples,
        p_values=STANDARD_P_VALUES,
        p_weights=STANDARD_P_WEIGHTS,
        verbose=True
    )

    # Save to disk
    cache.save(name)

    print(f"✅ Flat dataset '{name}' saved successfully!")

    # Cleanup
    del cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Flat dataset generation complete!")
print(f"{'='*60}")

#### List Existing Flat Datasets

In [ ]:
# List all cached flat datasets
datasets = FlatDatasetCache.list_datasets(base_path=BASE_PATH)

if not datasets:
    print("No cached flat datasets found.")
    print("Run the generation cells above to create some!")
else:
    print(f"Found {len(datasets)} cached flat dataset(s):\n")
    for ds in datasets:
        print(f"📁 {ds['name']}")
        print(f"   Samples: {ds.get('n_samples', 'unknown'):,}")
        print(f"   Distance: d={ds.get('d', '?')}")
        print(f"   Detectors: {ds.get('num_detectors', '?')}")
        print(f"   Error rates: {ds.get('p_values', '?')}")
        print(f"   Size: {ds.get('size_mb', 0):.1f} MB")
        print()

#### Training

Train SimpleNN models across multiple distances and seeds (like GraphSAGE baseline training).

In [ ]:
# Directory where models are saved
models_dir = BASE_PATH / "models" / "simple_nn"
models_dir.mkdir(parents=True, exist_ok=True)

# Distances to train (using cached flat datasets)
distances = [3, 5, 7, 9]
seeds = [1, 2, 3, 4, 5]

for d in distances:
    for seed in seeds:
        nickname = f"d{d}_baseline_seed_{seed}"
        cache_name = f"d{d}_baseline"

        # Check if model already exists
        existing = list(models_dir.glob(f"{nickname}_*.pt"))
        if existing:
            print(f"Model '{nickname}' already exists: {existing[0].name}, skipping...")
            continue

        # Check if flat dataset exists
        flat_data_path = BASE_PATH / "datasets" / "flat" / f"{cache_name}.pt"
        if not flat_data_path.exists():
            print(f"Flat dataset '{cache_name}' not found, skipping d={d}...")
            print(f"  Run the dataset generation cell first!")
            continue

        print(f"\n{'='*60}")
        print(f"Training {nickname}...")
        print(f"{'='*60}")

        trainer = SimpleNNTrainer(
            nickname=nickname,
            cache_name=cache_name,  # Use cached flat dataset
            epochs=10,
            batch_size=256,
            lr=1e-3,
            hidden_dim=256,
            base_path=BASE_PATH,
            device=device,
            seed=seed
        )

        # Train and save immediately
        model = trainer.train()
        model.save(nickname)
        print(f"Saved model: {nickname}")

        # Clean up to free RAM
        del model
        del trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Memory cleared.")

print(f"\nDone! Training complete.")